# 03 · Gold: point-in-time features and labels

**Pipeline:** bronze → silver → **gold features** → churn model → ARR forecast → retention playbook

One row per customer-month. The prediction is made **at the end of month *t* for month *t + 1***, so:

* every feature uses only rows at or before *t* (window frames end at `CURRENT ROW`; nothing reads
  `churn_date`) - verified by the point-in-time test in `tests/test_features.py`;
* `churn_next_month = 1` on a customer's **last paying month** (`churn_date = t + 1`). There is no row
  for the churn month itself, so the model never sees post-churn behaviour;
* rows in the **final observed month have a NULL label** - their outcome isn't known yet. They are
  excluded from training and are exactly the rows scored for the current risk snapshot.

New features vs. the original version: login trend vs. the prior quarter, logins vs. the customer's
own baseline, months to renewal, contract type, acquisition channel, 3-month MRR change and recent
downgrades.

In [1]:
%pip install -q -r ../requirements.txt

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from revenue_pipeline import config, spark_utils

spark = spark_utils.get_spark()
print(f"Spark {spark.version} | schema: {config.SCHEMA} | Databricks: {spark_utils.on_databricks()}")

Spark 3.5.3 | schema: revenue_forecasting | Databricks: False


In [3]:
from revenue_pipeline import features

gold = spark_utils.write_table(spark, features.build_gold_features(spark), "gold_customer_features")

summary = spark.sql('''
    SELECT COUNT(*)                                        AS rows,
           COUNT(DISTINCT customer_id)                     AS customers,
           SUM(CASE WHEN churn_next_month IS NULL THEN 1 END) AS unlabelled_snapshot_rows,
           AVG(churn_next_month)                           AS churn_label_rate
    FROM gold_customer_features''').toPandas()
print(summary.to_string(index=False))

  rows  customers  unlabelled_snapshot_rows  churn_label_rate
134732      10000                      8170          0.014388


**Sanity check - one churner's final months.** The label fires on the last paying month, and the
warning signs (logins falling below the customer's own baseline, more tickets) are visible *before*
the churn. Not every churner looks like this: some leave with no behavioural warning at all, which
is what caps the model's recall.

In [4]:
churner = spark.sql('''
    SELECT customer_id FROM gold_customer_features
    WHERE churn_next_month = 1 AND contract_annual = 0 AND tenure_months > 12
      AND logins_vs_baseline < 0.6 AND rolling_3m_tickets > 1
    ORDER BY customer_id LIMIT 1''').first()[0]
cols = ["month", "mrr", "logins", "logins_vs_baseline", "rolling_3m_tickets", "months_to_renewal", "churn_next_month"]
print(spark.table("gold_customer_features").filter(f"customer_id = {churner}")
      .orderBy("month").select(*cols).toPandas().tail(6).round(2).to_string(index=False))

     month  mrr  logins  logins_vs_baseline  rolling_3m_tickets  months_to_renewal  churn_next_month
2023-07-01 99.0      10                1.48                0.00                  1                 0
2023-08-01 99.0       5                0.72                0.33                  1                 0
2023-09-01 99.0      16                2.34                0.33                  1                 0
2023-10-01 99.0      10                1.37                1.00                  1                 0
2023-11-01 99.0       6                0.81                1.00                  1                 0
2023-12-01 99.0       3                0.41                1.33                  1                 1


In [5]:
monthly_arr = spark.sql('''
    SELECT CAST(month AS STRING) AS ds, SUM(arr) AS y
    FROM gold_customer_features GROUP BY month ORDER BY month''')
spark_utils.write_table(spark, monthly_arr, "gold_monthly_arr")
print(spark_utils.export_csv(spark, "gold_monthly_arr"))

/Users/madhumithachandrasekaran/revenue_forecasting/data/gold/gold_monthly_arr.csv
